## Script to find all orbits and download SWOT data from within a bounding box 

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import os
import geopandas as gpd
import numpy as np
from shapely import geometry
import pdb

import cartopy.crs as ccrs
import matplotlib.pyplot as plt
import matplotlib.patches as patches
#%pip install earthaccess
import earthaccess
import os
import re

In [ ]:
%pip install speedtest-cli
import speedtest

def check_internet_speed():
    """
    Checks and prints the internet download and upload speed.
    """
    st = speedtest.Speedtest()
    print("Getting best server...")
    st.get_best_server()
    print(f"Testing download speed...")
    download_speed = st.download()
    print(f"Testing upload speed...")
    upload_speed = st.upload()

    # Convert speeds from bits per second to megabits per second (Mbps)
    download_mbps = download_speed / 1_000_000
    upload_mbps = upload_speed / 1_000_000

    print(f"\nDownload Speed: {download_mbps:.2f} Mbps")
    print(f"Upload Speed: {upload_mbps:.2f} Mbps")
    print(f"Ping: {st.results.ping:.2f} ms")

check_internet_speed()

In [ ]:
os.chdir('/g/data/jk72/jl1058/download_swot')

## Functions 

In [ ]:
def make_bboxes_from_180(lon_min, lon_max, lat_min, lat_max):

    lon_range_180 = (lon_min, lon_max)
    bbox_180 = (lon_min, lat_min, lon_max, lat_max)

    # --- GLOBAL CASE ---
    if lon_min == -180 and lon_max == 180:
        return lon_range_180, bbox_180, [(0, lat_min, 360, lat_max)]

    lon_min_360 = lon_min % 360
    lon_max_360 = lon_max % 360

    crosses_dateline = lon_min > lon_max

    bboxes_360 = []

    if not crosses_dateline:
        bboxes_360.append((lon_min_360, lat_min, lon_max_360, lat_max))
    else:
        bboxes_360.append((lon_min_360, lat_min, 360, lat_max))
        bboxes_360.append((0, lat_min, lon_max_360, lat_max))

    return bbox_180, bboxes_360
def get_half_orbits_intersect(bbox_180, bbox_360):
    """
    Get half orbits that intersect either a -180–180 or 0–360 bounding box.

    Parameters
    ----------
    bbox_180 : tuple or list
        (minlon, minlat, maxlon, maxlat) in -180..180 convention
    bbox_360 : tuple or list
        (minlon, minlat, maxlon, maxlat) in 0..360 convention

    Returns
    -------
    gpd.GeoDataFrame
        Half-orbits intersecting either bbox, with intersection area.
    """
    swath_geometries = gpd.read_file(GEOMETRIES_FILE)

    poly_180 = geometry.box(*bbox_180)
    poly_360 = geometry.box(*bbox_360)

    def _compute_intersection_area(row):
        geom = row.geometry

        area_180 = 0.0
        area_360 = 0.0

        if geom.intersects(poly_180):
            area_180 = geom.intersection(poly_180).area

        if geom.intersects(poly_360):
            area_360 = geom.intersection(poly_360).area

        # keep the larger intersection
        return max(area_180, area_360)

    swath_geometries["intersection_area"] = swath_geometries.apply(
        _compute_intersection_area, axis=1
    )
    swath_geometries = swath_geometries[swath_geometries["intersection_area"] > area_threshold]
    return swath_geometries



def plot_geometries(geometries, lon_range, lat_range, title, plot_extent=None):
    fig, ax = plt.subplots(
        ncols=1,
        figsize=(12, 8),
        subplot_kw={"projection": ccrs.PlateCarree()},
    )

    gpd.GeoSeries(geometries.geometry).plot(
        ax=ax,
        transform=ccrs.PlateCarree(),
        alpha=1,
    )

    square = patches.Rectangle(
        (lon_range[0], lat_range[0]),
        lon_range[1] - lon_range[0],
        lat_range[1] - lat_range[0],
        edgecolor="orange",
        facecolor="none",
        transform=ccrs.PlateCarree(),
    )
    ax.add_patch(square)

    ax.set_title(title)
    ax.coastlines()

    if plot_extent:
        ax.set_extent(plot_extent, crs=ccrs.PlateCarree())


def delete_unmatched_pass_files(directory, filtered, dry_run=True):
    """
    Print (dry run) or delete files whose pass number is not in the filtered swath_geometries.

    Parameters
    ----------
    directory : str
        Path to directory containing SWOT .nc files
    filtered : GeoDataFrame
        Filtered GeoDataFrame containing valid half-orbit pass numbers.
        Must include a column "pass_number" (int or str)
    dry_run : bool
        If True, only print what would be deleted. If False, delete the files.
    """

    # Convert all pass numbers in filtered to integers
    valid_passes = set(filtered["pass_number"].astype(int))

    for filename in os.listdir(directory):
        if not filename.endswith(".nc"):
            continue

        parts = filename.split("_")
        if len(parts) < 7:
            continue  # skip malformed filenames

        # Extract pass number from filename
        try:
            pass_num = int(parts[6])  # 7th field is the pass number
        except ValueError:
            continue

        # Delete if pass number is NOT in filtered
        if pass_num not in valid_passes:
            file_path = os.path.join(directory, filename)

            if dry_run:
                print(f"[DRY RUN] Would delete: {file_path}")
            else:
                print(f"Deleting: {file_path}")
                os.remove(file_path)


def define_bbox(bbox_name):
    if bbox_name == "Mawson Coast":
        lon_range_180 = (50, 75)
        lat_range = (-70, -60)

    elif bbox_name == "Kapp Norvegia":
        lon_range_180 = (-18, -3)
        lat_range = (-73, -68)

    elif bbox_name == "Svalbard":
        lon_range_180 = (8, 28)
        lat_range = (76, 82)

    elif bbox_name == "Thwaites":
        lon_range_180 = (-113, -100)
        lat_range = (-75.5, -72)

    elif bbox_name == "Amery":
        lon_range_180 = (69, 75)
        lat_range = (-71, -68.5)
        
    elif bbox_name == "Emily":
        lon_range_180 = (-65.01, -52.86)
        lat_range = (-66.3, -61.14)
    elif bbox_name == "Victoria_Land":
        lon_range_180 = (161, 179.5)
        lat_range = (-78.02, -70.41)
    elif bbox_name == "Atka Bay":
        lon_range_180 = (-9, -6)
        lat_range = (-71, -70)
    elif bbox_name == "Penn Test":
        lon_range_180 = (-70, -60)
        lat_range = (-76, -69)
    elif bbox_name == "orb prob1":
        lon_range_180 = (-20.713,-24.672)
        lat_range = (-72.812, -74.668)
        

    elif bbox_name == "ALL":
        lon_range_180 = (-180, 180)
        lat_range = (-81, -60)
        # --- circumpolar special case ---
        lon_min, lon_max = -180, 180
        lat_min, lat_max = -81, -60

        lon_range_180 = (lon_min, lon_max)
        lon_range_360 = (0, 360)
        lat_range = (lat_min, lat_max)
        bbox_180 = [lon_min, lat_min, lon_max, lat_max]
        bbox_360 = [0, lat_min, 360, lat_max] # always a single 0–360 box
        plot_extent_180 = [-180, 180, -90, -40]
        plot_extent_360 = [0, 360, -90, -40]

        return (
            lon_range_180,
            lon_range_360,
            lat_range,
            bbox_180,
            bbox_360,
            plot_extent_180,
            plot_extent_360,
        )

    else:
        raise ValueError(f"Bounding Box Region Not Defined: {bbox_name}")

    # --- derive 0–360 longitude version ---
    lon_range_360 = tuple(l + 360 if l < 0 else l for l in lon_range_180)

    # --- bounding boxes ---
    bbox_180 = [
        lon_range_180[0],
        lat_range[0],
        lon_range_180[1],
        lat_range[1],
    ]

    bbox_360 = [
        lon_range_360[0],
        lat_range[0],
        lon_range_360[1],
        lat_range[1],
    ]

    plot_extent_180 = [
        lon_range_180[0] - 40,
        lon_range_180[1] + 40,
        lat_range[0] - 25,
        lat_range[1] + 25,
    ]

    plot_extent_360 = [
        lon_range_360[0] - 40,
        lon_range_360[1] + 40,
        lat_range[0] - 25,
        lat_range[1] + 25,
    ]

    return (
        lon_range_180,
        lon_range_360,
        lat_range,
        bbox_180,
        bbox_360,
        plot_extent_180,
        plot_extent_360,
    )

In [ ]:
lon_range_180, lon_range_360, lat_range, bbox_180, bbox_360, ext_180, ext_360 = define_bbox("orb prob1")

In [ ]:
bbox_360

## Define Time

In [ ]:
T0 = "2025-12-01"

T1 = "2025-12-31"

In [ ]:
area_threshold = 1 # Mawson Coast, Svalbard
# area_threshold = 15 # Kapp Norvegia
phase = 'science'
GEOMETRIES_FILE = f'KaRIn_2kms_{phase}_geometries.geojson'
swath_geoms = get_half_orbits_intersect(bbox_180, bbox_360)
#swath_geoms.to_file("/g/data/jk72/ae8456/SWOT/swath_geoms.shp")

In [ ]:
passes_to_plot2 = swath_geoms['pass_number'].values

In [ ]:
#passes_to_plot2 = [
#   18,  27,  46, 277, 296, 305, 324, 583]
passes_to_plot = list(range(1, 584))
swath_subset = swath_geoms[swath_geoms['pass_number'].isin(passes_to_plot)]
swath_subset2 = swath_geoms[swath_geoms['pass_number'].isin(passes_to_plot2)]
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 10))

swath_subset.plot(
    ax=ax,
    column='pass_number',
    legend=True,
    linewidth=1
)
swath_subset2.plot(
    ax=ax,
    column='pass_number',
    legend=True,
    linewidth=1,
    color='black'
)

ax.set_title("Selected SWOT Passes")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")

plt.show()


In [ ]:
swath_geoms_filt = swath_geoms[swath_geoms["intersection_area"] > area_threshold]
plot_geometries(swath_geoms_filt, lon_range_180, lat_range, f"Swot {phase} Geometries", ext_180)

In [ ]:
swath_geoms_filt

In [ ]:
# science, calval
phase = 'science'
GEOMETRIES_FILE = f'KaRIn_2kms_{phase}_geometries.geojson'

In [ ]:
swath_geoms = get_half_orbits_intersect(bbox_180, bbox_360)


In [ ]:
passes = swath_geoms['pass_number'].sort_values().unique()

expected = np.arange(passes.min(), passes.max() + 1)

missing_passes = np.setdiff1d(expected, passes)

print("Missing passes:", missing_passes)

In [ ]:
assert (swath_geoms.pass_number % 2 == 1).any(), "No odd pass numbers found"
assert (swath_geoms.pass_number % 2 == 0).any(), "No even pass numbers found"

In [ ]:
# swath_geometries is your GeoDataFrame with an 'intersection_area' column
values = swath_geoms["intersection_area"].dropna()

plt.figure(figsize=(8, 5))
plt.hist(values, bins=30)   # No color/style specified (per requirements)
plt.xlabel("Intersection Area")
plt.ylabel("Frequency")
plt.title("Histogram of Intersection Area")
plt.show()

In [ ]:
half_orbits = list(swath_geoms_filt['pass_number'])

In [ ]:
earthaccess.login()

In [ ]:
os.chdir('/g/data/gv90/ae8456/SWOT_SEAICE/DATA')

## L2 Unsmoothed

In [ ]:
strep = f"{half_orbits[0]:03}"
gran_name =  f"*Unsmoothed_*_{strep}_*"
print(gran_name)

In [ ]:
results = earthaccess.search_data(
    #short_name="SWOT_L2_LR_SSH_2.0",
    short_name="SWOT_L2_LR_SSH_D",
    granule_name = gran_name,
    orbit_number = [],
    temporal=(T0,T1),
    bounding_box=(lon_range_180[0],lat_range[0],lon_range_180[1], lat_range[1]),
    )

print(len(results))

In [ ]:
results_str = " ".join(map(str, results))
# Regex pattern to match URLs inside Data: [...]
pattern = r"Data:\s*\[(.*?)\]"
# Find all matches
matches = re.findall(pattern, results_str)
# Each match may contain multiple URLs separated by commas
all_urls = []
for match in matches:
    # Remove quotes and whitespace, split by comma
    urls = [url.strip().strip("'\"") for url in match.split(",")]
    all_urls.extend(urls)
# print(all_urls)
filenames = [url.split("/")[-1] for url in all_urls]
print(filenames)

In [ ]:
data_dir = "SWOT_L2_Unsmoothed_Full"
os.makedirs(data_dir, exist_ok=True)  
# Loop over all half orbits

for half_orbit in half_orbits[0:]:
    strep = f"{half_orbit:03}"
    gran_name = f"*Unsmoothed_*_{strep}_*"
    print(f"Searching for granule: {gran_name}")
    # Thwaites Glacier
    results = earthaccess.search_data(
        # short_name="SWOT_L2_LR_SSH_2.0", # version C
        short_name="SWOT_L2_LR_SSH_D",
        granule_name = gran_name,
        orbit_number = [],
        temporal=(T0,T1),
        bounding_box=(lon_range_180[0],lat_range[0],lon_range_180[1], lat_range[1]),
        )
    print(f"Found {len(results)} results for half_orbit {half_orbit}")

    # Convert the results list to a single string
    results_str = " ".join(map(str, results))
    # Regex pattern to match URLs inside Data: [...]
    pattern = r"Data:\s*\[(.*?)\]"
    # Find all matches
    matches = re.findall(pattern, results_str)
    # Each match may contain multiple URLs separated by commas
    all_urls = []
    for match in matches:
        # Remove quotes and whitespace, split by comma
        urls = [url.strip().strip("'\"") for url in match.split(",")]
        all_urls.extend(urls)
    # print(all_urls)
    filenames = [url.split("/")[-1] for url in all_urls]

    # list of existing files in directory
    dir_files = os.listdir(data_dir)
    
    mask = []
    
    for fname in filenames:
        # extract the identifying part (before the timestamp)
        key = "_".join(fname.split("_")[:7])
        print(key)
        # example result: SWOT_L2_LR_SSH_Unsmoothed_012_355
    
        # check if any file in directory contains this key
        exists = any(key in f for f in dir_files)
    
        # 0 = file exists, 1 = no file
        mask.append(0 if exists else 1)

    # OPTIONAL: don't download a file with any version of the same file already in data_dir
    filtered_results = [r for r, m in zip(results, mask) if m == 1]

    # if results:
    if filtered_results:
        files = earthaccess.download(
            filtered_results,
            local_path=data_dir,
            provider="POCLOUD",
            threads=4
        )
        print(f"Downloaded {len(files)} files for half_orbit {half_orbit}")
    else:
        print(f"No files found for half_orbit {half_orbit}")

## L2 Expert

In [ ]:
# Loop over all half orbits

for half_orbit in half_orbits[0:]:
    strep = f"{half_orbit:03}"
    gran_name = f"*Expert_*_{strep}_*"
    print(f"Searching for granule: {gran_name}")
 
    results = earthaccess.search_data(
        #short_name="SWOT_L2_LR_SSH_2.0",
        short_name="SWOT_L2_LR_SSH_D",
        granule_name = gran_name,
        orbit_number = [],
        temporal=(T0,T1),
        bounding_box=(lon_range_180[0],lat_range[0],lon_range_180[1], lat_range[1]),
        )
    
    print(f"Found {len(results)} results for half_orbit {half_orbit}")

    if results:
        files = earthaccess.download(
            results,
            local_path="SWOT_L2_Expert_Full",
            provider="POCLOUD",
            threads=4
        )
        print(f"Downloaded {len(files)} files for half_orbit {half_orbit}")
    else:
        print(f"No files found for half_orbit {half_orbit}")

## L3

In [ ]:
import os
from getpass import getpass
import ftplib
from urllib.parse import urlparse
import xarray as xr
from concurrent.futures import ThreadPoolExecutor, as_completed
import cartopy.crs as ccrs
import matplotlib.pyplot as plt
os.chdir('/g/data/jk72/ae8456/SWOT')
# import swot_helpers

In [ ]:
import re
import time

def _download_file_thread(ftp_details, ftp_path, filename, target_directory, max_retries=3):
    """Download a single file with a new FTP connection (thread-safe),
    skip if a file with same _CCC_HHH_ exists, retry on failure.
    """
    ftp_host, username, password = ftp_details
    local_filepath = os.path.join(target_directory, filename)

    # ------------------------------------------------------------------
    # Extract the cycle_half_orbit key, e.g. "_016_551_"
    match = re.search(r'_(\d{3})_(\d{3})_', filename)
    if match:
        key = f"_{match.group(1)}_{match.group(2)}_"
    else:
        key = None
    # ------------------------------------------------------------------

    # Skip if any file in target_directory contains the same key
    if key is not None:
        for existing_file in os.listdir(target_directory):
            if key in existing_file:
                print(f"Skipping {filename}, matching file already exists: {existing_file}")
                return os.path.join(target_directory, existing_file)

    # ------------------------------------------------------------------
    # Download logic
    for attempt in range(1, max_retries + 1):
        try:
            print(f"Downloading {filename}, attempt {attempt}")
            with ftplib.FTP(ftp_host) as ftp:
                ftp.login(username, password)
                ftp.cwd(ftp_path)
                with open(local_filepath, 'wb') as file:
                    ftp.retrbinary(f'RETR {filename}', file.write)
            print(f"Downloaded {filename}")
            return local_filepath

        except ftplib.error_perm as e:
            if '530' in str(e):
                print(f"FTP session limit reached, waiting before retrying {filename}...")
                time.sleep(10)
            else:
                print(f"Permission error downloading {filename}: {e}")
                break

        except Exception as e:
            print(f"Error downloading {filename} (attempt {attempt}): {e}")
            time.sleep(5)

    print(f"Failed to download {filename} after {max_retries} attempts.")
    return None

# --------------------------------------------------------------------------- # 

def _get_last_version_filename(filenames):
    versions = {int(f[-5:-3]): f for f in filenames}
    return versions[max(versions.keys())]

# --------------------------------------------------------------------------- # 

def _select_filename(filenames, only_last):
    if not only_last: return filenames
    return [_get_last_version_filename(filenames)]

# --------------------------------------------------------------------------- # 

def ftp_download_files_parallel(ftp_path, level, variant, cycle_numbers, half_orbits, output_dir, only_last=True, max_workers=4):
    """Download half orbit files in parallel from AVISO's FTP server."""
    ftpAVISO = 'ftp-access.aviso.altimetry.fr'
    ftp_details = (ftpAVISO, username, password)  # Use your FTP credentials here

    downloaded_files = []

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = []

        for cycle in cycle_numbers:
            cycle_str = f'{cycle:03d}'
            cycle_dir = f'cycle_{cycle_str}'
            full_cycle_path = os.path.join(ftp_path, cycle_dir)

            for half_orbit in half_orbits:
                half_orbit_str = f'{half_orbit:03d}'
                pattern = f'SWOT_{level}_LR_SSH_{variant}_{cycle_str}_{half_orbit_str}'

                # Create a temporary FTP connection to list files
                try:
                    with ftplib.FTP(ftpAVISO) as ftp:
                        ftp.login(username, password)
                        ftp.cwd(full_cycle_path)
                        filenames = ftp.nlst(f'{pattern}_*')
                        if level == "L3":
                            only_last_flag = False
                        else:
                            only_last_flag = only_last
                        filenames = _select_filename(filenames, only_last_flag)
                except Exception as e:
                    print(f"No pass {half_orbit} in cycle {cycle}: {e}")
                    filenames = []

                for f in filenames:
                    futures.append(
                        executor.submit(_download_file_thread, ftp_details, full_cycle_path, f, output_dir, 3)
                    )

        for future in as_completed(futures):
            result = future.result()
            if result:
                downloaded_files.append(result)

    return downloaded_files

# --------------------------------------------------------------------------- # 

def _normalized_ds(ds, lon_min, lon_max):
    lon = ds.longitude.values
    lon[lon < lon_min] += 360
    lon[lon > lon_max] -= 360
    ds.longitude.values = lon
    return ds

# --------------------------------------------------------------------------- # 

def _subset_ds(file, variables, lon_range, lat_range, output_dir):
    print(f"Subset dataset: {file}")
    swot_ds = xr.open_dataset(file)
    swot_ds = swot_ds[variables]
    swot_ds.load()
    
    ds = _normalized_ds(swot_ds.copy(), -180, 180)

    mask = (
        (ds.longitude <= lon_range[1])
        & (ds.longitude >= lon_range[0])
        & (ds.latitude <= lat_range[1])
        & (ds.latitude >= lat_range[0])
    ).compute()
    
    swot_ds_area = swot_ds.where(mask, drop=True)

    if swot_ds_area.sizes['num_lines'] == 0:
        print(f'Dataset {file} not matching geographical area.')
        return None

    for var in list(swot_ds_area.keys()):
        swot_ds_area[var].encoding = {'zlib':True, 'complevel':5}

    filename = f"subset_{os.path.basename(urlparse(file).path)}"
    
    print(f"Store subset: {filename}")
    filepath = os.path.join(output_dir, filename)
    swot_ds_area.to_netcdf(filepath, mode='w')
        
    return filepath

# --------------------------------------------------------------------------- # 

def subset_files_parallel(filenames, variables, lon_range, lat_range, output_dir, max_workers=4):
    """Subset datasets in parallel with a given geographical area."""
    subsetted_files = []

    # Submit tasks to the executor
    with ProcessPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(_subset_ds, f, variables, lon_range, lat_range, output_dir): f for f in filenames}

        for future in as_completed(futures):
            result = future.result()
            if result is not None:
                subsetted_files.append(result)

    return subsetted_files

# --------------------------------------------------------------------------- # 

def plot_datasets(filenames, variable, extent=None):
    cb_args = dict(
        add_colorbar=True,
        cbar_kwargs={"shrink": 0.3}
    )
    
    plot_kwargs = dict(
        x="longitude",
        y="latitude",
        cmap="Spectral_r",
        vmin=-0.2,
        vmax=0.2,
    )

    fig, ax = plt.subplots(figsize=(12, 8), subplot_kw=dict(projection=ccrs.PlateCarree()))
    if extent: ax.set_extent(extent)
    
    for filename in filenames:
        ds = xr.open_dataset(filename)
        ds[variable].plot.pcolormesh(
            ax=ax,
            **plot_kwargs,
            **cb_args)
        cb_args=dict(add_colorbar=False)
    
    ax.coastlines()
    ax.gridlines(draw_labels=True)




In [ ]:
from pathlib import Path

# Where you want the L3 Unsmoothed files to go
L3_OUT_DIR = Path("/g/data/gv90/ae8456/SWOT_SEAICE/DATA/SWOT_L3_Unsmoothed_Full/")
L3_OUT_DIR.mkdir(parents=True, exist_ok=True)

output_dir = L3_OUT_DIR

In [ ]:
# AVISO+ credentials (export these in your environment before running)
username = "jack.landy@bristol.ac.uk"
password = "REDACTED_USE_ENVIRONMENT"

In [ ]:
### SELECT VERSION ###
#ftp_path = '/swot_products/l3_karin_nadir/l3_lr_ssh/v2_0_1/Unsmoothed/'
ftp_path = '/swot_products/l3_karin_nadir/l3_lr_ssh/v3_0/Unsmoothed/forward/'
# ftp_path = '/swot_products/l3_karin_nadir/l3_lr_ssh/v1_0_2/Unsmoothed/'

# For selecting files with a regex pattern
# Basic, Expert, WindWave, Unsmoothed
variant = "Unsmoothed"
# L2, L3
level = "L3"

In [ ]:
# Define cycles and half_orbits numbers to download
# cycle_numbers = list(range(11, 30)) # Mawson Coast
cycle_numbers = list(range(43,44))
pass_numbers = half_orbits

print(cycle_numbers)

In [ ]:
downloaded_files = ftp_download_files_parallel(ftp_path, level, variant, cycle_numbers, pass_numbers, output_dir, only_last=True, max_workers=3)